# Demonstrating the PathSmoother class

The aim of this tutorial is to showcase the functionality of the PathSmoother, a class designed to take either a trajectory from a NanoVer simulation or a set of coordinates defining the path of an atom or group of atoms (and their atomic masses), and produce a smoothed trajectory of equally spaced points (i.e. a constant speed trajectory) to define the path of an iGUESSMD force.

Currently this class is designed to extract paths defined in the absolute space of the simulation, which works in the case when part of the system has been constrained/restrained to generate a meaningful real-space path.

In [ ]:
from nanover.iguessmd.pathsmoothing import PathSmoother

## Creating a PathSmoother...

### ...from a NanoVer recording

A PathSmoother object can be created directly from a NanoVer recording. In this case, the PathSmoother will look for the first user force applied to the simulation and the atom(s) to which it was applied, and extract the path that the atom(s) took during the application of the force. It will also calculate the path of the centre of mass (COM), which is what it will use to generate a constant speed path. For a single atom, the trajectory of the atom and the COM are equivalent.

In [ ]:
# Create the PathSmoother from a NanoVer trajectory file
ps = PathSmoother.from_recording(filename="../systems/recordings/iguessmd_deca-alanine.nanover.zip")

In [ ]:
print(f"Indices of the atoms defining the trajectory:\n{ps.atom_indices}\n")
print(f"Positions of the atoms defining the trajectory / nm:\n{ps.atom_positions}\n")
print(f"Masses of the atoms defining the trajectory / Da:\n{ps.all_atom_masses[ps.atom_indices]}\n")

### ...from a specified trajectory and set of atom masses

It is also possible to generate a PathSmoother object by directly passing the trajectories of a set of atoms. In this case, it is important to also pass the masses of the atoms to the PathSmoother, such that the trajectory of the COM can be calculated. In this case, the indices defining the atoms are redundant, so do not need to be passed to the PathSmoother.

In [ ]:
ps_from_coords = PathSmoother.from_path_coordinates(coords=ps.atom_positions, atom_masses=ps.all_atom_masses[ps.atom_indices])

## Plotting trajectories of the atoms and COM

The PathSmoother has functions to plot the trajectories of the atoms and the trajectory of the COM. When used in a Jupyter notebook, these plots are interactive.

When plotting the trajectories of the atoms, we can optionally ask the PathSmoother to plot the initial positions of a selection of the atoms in the system. This can be useful to help understand the trajectory in the context of the system. Selections are made using MDAnalysis-style selection strings

The PathSmoother class can detect both atoms the user interacted with during the first iMD interaction and any atoms whose positions are constrained in the simulation. If interacted or constrained atoms appear in the selection of positions to plot defined by the user, they will be highlighted in orange or blue, respectively.

In [ ]:
# Define atoms to plot initial positions for [Optional]
plot_initial_positions = True
initial_position_selection = 'all'

# Plot trajectories of atoms
ps.plot_atoms_trajectories(plot_atom_positions=plot_initial_positions, atom_selection=initial_position_selection)

After opening an interactive plot, it is good practice to close it before opening another. The PathSmoother class has a function that can perform this cleanup:

In [ ]:
# Close open interactive plots
ps.close_interactive_plots()

By default, the plots use the viridis colour map, and the axes adapt to better represent the trajectory through 3-D space. It is possible to specify whether the plots should maintain an equal aspect ratio, and to change the color map defining the progression of the trajectory (to any of Matplotlib's available colour maps, [see here](https://matplotlib.org/stable/users/explain/colors/colormaps.html):

In [ ]:
# Plot trajectory of the COM
ps.plot_com_trajectory(equal_aspect_ratio=True,
                       cmap='plasma',
                       plot_atom_positions=plot_initial_positions,
                       atom_selection=initial_position_selection)

In [ ]:
# Close open interactive plots
ps.close_interactive_plots()

## Smoothing the COM trajectory

The main benefit of the PathSmoother class is the ability to interactively refine the path given to the PathSmoother. In order to do this, a curve is fitted through the points, which can be edited to change:
- the initial point at which the trajectory begins
- the final point at which the trajectory ends
- the smoothness of the curve defining the refined path
- the number of points used to fit the refined path (approximately equally spaced)
- the alpha value of the original points defining the trajectory

To see this in action, run the cell below and adjust the sliders to see the effect on the smoothed trajectory (plotted in full colour) with respect to the original path defined by the trajectory passed to the PathSmoother (transparent points)

In [ ]:
ps.create_interactive_smoothing_plot(plot_atom_positions=plot_initial_positions,
                                     atom_selection=initial_position_selection,
                                     cmap='plasma')

Once you have defined the path as you want it to be, run the function `save_smoothing_plot_result()`, which save the adjusted parameters within the PathSmoother class. The parameters define the curve from which the constant speed trajectory will be produced.

In [ ]:
ps.save_smoothing_plot_result()
ps.close_interactive_plots()

You can then check the curve you produced by running the cell below:

In [ ]:
ps.plot_smoothed_com_trajectory(equal_aspect_ratio=False,
                                plot_atom_positions=plot_initial_positions,
                                atom_selection=initial_position_selection)

In [ ]:
ps.close_interactive_plots()

## Generating the constant speed trajectory

Now everything is in place to be able to calculate the constant speed trajectory along the curve defining the smoothed path of the COM of the atom(s). The PathSmoother has an inbuilt function to do this, which requires definition of the desired speed (in nm/ps) and timestep of the iGUESSMD simulation for which the constant speed trajectory will define the path of the iGUESSMD force.

In [ ]:
desired_speed_nm_ps = 0.01
desired_timestep_ps = 0.002
ps.calculate_constant_speed_trajectory(desired_speed_nm_ps=desired_speed_nm_ps, timestep_ps=desired_timestep_ps)

It is important to check that the calculation completed successfully. In some cases, it may be necessary to explicitly adjust the convergence criteria. If the calculation does not converge, then the results cannot be saved to an output file.

In [ ]:
ps.calculate_constant_speed_trajectory(desired_speed_nm_ps=desired_speed_nm_ps, timestep_ps=desired_timestep_ps, rel_tolerance=1e-4, max_iterations=20)

## Plotting and saving the constant speed trajectory

The final steps now are to check the appearance of the constant speed trajectory by plotting it, and save the results to an output file.

In [ ]:
ps.plot_constant_speed_trajectory(plot_atom_positions=plot_initial_positions,
                                  atom_selection=initial_position_selection)

In [ ]:
ps.close_interactive_plots()

The data for the constant speed trajectory is saved as compressed NumPy arrays, and the file extension of the output file should be ".npz". The following data are saved:
- the coordinates defining the constant speed trajectory
- the constant speed that the trajectory defines, in nm/ps
- the timestep of the iGUESSMD simulation that the trajectory is designed to be used with, in ps
- (if available) the indices defining the atoms to which the iGUESSMD force should be applied

In [ ]:
# Save the constant speed trajectory generated by the PathSmoother
outfile = "./deca-alanine_refined_iguessmd_path.npz"
ps.save_constant_speed_trajectory_data(output_filepath=outfile)

This data can then be loaded for use in an iGUESSMD simulation as demonstrated below:

In [ ]:
import numpy as np

loaded_iguessmd_path_data = dict(np.load(outfile))
loaded_iguessmd_path_data.keys()

This data can also be loaded using a built-in function provided by the iGUESSMD module, which returns a dictionary containing the information above (if available)

In [ ]:
from nanover.iguessmd.pathsmoothing import load_iguessmd_path_data

# Returns a dictionary of the PathSmoother results
ps_results = load_iguessmd_path_data(outfile)
ps_results.keys()